# SAHIBEX — free GPU model smoke test (Kaggle)

Run this notebook in a Kaggle GPU session. Upload `Qwen3-4B-Q4_K_M.gguf` as a **private Kaggle Dataset** first, then attach that dataset to this notebook. This notebook only starts a local test server and checks a simple answer; it does not expose a public endpoint or change your phone.

GPU access and session quota are controlled by Kaggle and may not be available on every account.

In [ ]:
import os, glob, subprocess, pathlib, time, json, urllib.request
print('GPU check:')
subprocess.run(['bash','-lc','nvidia-smi'], check=False)
matches = glob.glob('/kaggle/input/**/Qwen3-4B-Q4_K_M.gguf', recursive=True)
if not matches:
    raise FileNotFoundError('Attach your private dataset containing Qwen3-4B-Q4_K_M.gguf in Notebook > Add Input, then rerun this cell.')
MODEL = matches[0]
print('Model:', MODEL, 'size_GB:', round(os.path.getsize(MODEL)/1e9, 2))


In [ ]:
# Build llama.cpp with CUDA enabled for this GPU session. Build may take several minutes.
import os, subprocess, pathlib
if not pathlib.Path('/kaggle/working/llama.cpp/build/bin/llama-server').exists():
    subprocess.run('cd /kaggle/working && git clone --depth 1 https://github.com/ggml-org/llama.cpp.git', shell=True, check=True)
    subprocess.run('cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release', shell=True, check=True)
    subprocess.run('cmake --build /kaggle/working/llama.cpp/build --config Release -j 2 --target llama-server', shell=True, check=True)
print('llama-server ready')


In [ ]:
# Start on loopback only; do not expose a public port.
import subprocess, time, urllib.request, json, os
log = open('/kaggle/working/llama-server.log','w')
server = subprocess.Popen(['/kaggle/working/llama.cpp/build/bin/llama-server','-m',MODEL,'--host','127.0.0.1','--port','8080','-c','2048','--parallel','1','--n-gpu-layers','99','--threads','2'], stdout=log, stderr=subprocess.STDOUT)
healthy = False
for _ in range(90):
    if server.poll() is not None: break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=2) as r:
            healthy = r.status == 200
            if healthy: break
    except Exception: pass
    time.sleep(2)
if not healthy:
    print(open('/kaggle/working/llama-server.log').read()[-12000:])
    raise RuntimeError('Model server failed health check; inspect log above.')
print('SERVER HEALTH PASS; PID', server.pid)


In [ ]:
# Deterministic smoke test. Expected assistant content: 4
payload = {"model": MODEL, "messages": [{"role":"user","content":"Solve 2 + 2. Do not explain. Answer with just the number."}], "temperature": 0, "max_tokens": 80, "chat_template_kwargs": {"enable_thinking": False}}
req = urllib.request.Request('http://127.0.0.1:8080/v1/chat/completions', data=json.dumps(payload).encode(), headers={'Content-Type':'application/json'})
with urllib.request.urlopen(req, timeout=90) as r: result = json.load(r)
answer = result['choices'][0]['message'].get('content','').strip()
print('MODEL:', result.get('model'))
print('ANSWER:', repr(answer))
print('TOKENS:', result.get('usage'))
assert answer == '4', f'Expected exact answer 4, got {answer!r}'
print('INFERENCE SMOKE TEST: PASS')


## Current limits
This confirms only that the model can load and answer inside the Kaggle session. It does **not** yet connect SAHIBEX on Android to the cloud. Do not publish the notebook server or use an unauthenticated public tunnel. Next step after this passes is to design an authenticated, temporary connection and verify end-to-end requests. Stop the Kaggle session when finished to avoid consuming quota.